In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
import os

print("--- 1. Loading raw data ---")
df = pd.read_csv('../data/raw/cs-training.csv')

print("--- 2. Engineering API-Compliant Features ---")
df['Income_Annual'] = df['MonthlyIncome'] * 12
df['Spending_Ratio'] = df['DebtRatio']
df['Utility_Bill_Late_Count'] = (
    df['NumberOfTime30-59DaysPastDueNotWorse'] + 
    df['NumberOfTime60-89DaysPastDueNotWorse'] + 
    df['NumberOfTimes90DaysLate']
)
df['Credit_History_Length_Months'] = (df['age'] * 12) - 216
df.loc[df['Credit_History_Length_Months'] < 0, 'Credit_History_Length_Months'] = 0

# Breaking Perfect Colinearity
np.random.seed(42)
savings_multipliers = np.random.uniform(0.0, 6.0, size=len(df))
df['Savings_Balance'] = df['MonthlyIncome'] * savings_multipliers

df['Default'] = df['SeriousDlqin2yrs']

print("--- 3. Handling Missing Values safely ---")
feature_cols = [
    'Income_Annual', 'Savings_Balance', 'Spending_Ratio', 
    'Utility_Bill_Late_Count', 'Credit_History_Length_Months'
]

# Do NOT impute the Target variable ('Default'). Only impute features.
df_clean = df[feature_cols + ['Default']].copy()

imputer = SimpleImputer(strategy='median')
df_clean[feature_cols] = imputer.fit_transform(df_clean[feature_cols])

print("--- 4. Capping Outliers ---")
outlier_columns = ['Income_Annual', 'Spending_Ratio', 'Savings_Balance']
for col in outlier_columns:
    upper_limit = df_clean[col].quantile(0.99)
    df_clean[col] = df_clean[col].clip(upper=upper_limit)

print(f"✅ Success! {len(df_clean)} records ready.")
os.makedirs('../data/processed/', exist_ok=True)
df_clean.to_csv('../data/processed/clean_features.csv', index=False)

Loading raw data...
Engineering features...
Imputing missing values with medians...
Capping massive outliers at the 99th percentile...
Success! 150000 API-compliant records ready.
